[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_4_HigherOrder_Operators_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 📐 اللاب 4: مؤثرات الرتب العليا ومعادلة لابل التوافقية
### *غرفة التحكم: الأصل التحليلي لمعادلة لابل المستمرة*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الباب الأول، القسم 1.4 (صفحات 29–45)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $(D_x^2 + D_y^2)z = 0$ | "معادلة لابل: حقل جهد ناعم بدون قمم أو حفر محلية." | `laplace_continuous_eq` |
| $m^2 + 1 = 0 \implies m = \pm i$ | "الجذور المميزة المركبة: تثبت أن الجهد يتكون من دوال توافقية." | `roots = [1j, -1j]` |
| $z = \phi_1(y + ix) + \phi_2(y - ix)$ | "الحل العام التحليلي لمعادلة لابل في المستوى." | `analytic_harmonic_field` |

---
## 1. المعنى البسيط: "الغشاء المطاطي المشدود"
تخيل غشاءً مطاطياً مشدوداً بإحكام فوق إطار سلكي. إذا ضغطت على أحد أطراف الإطار للأعلى والطرف الآخر للأسفل:
* المطاط سيتشكل بسلاسة تامة.
* يستحيل أن تتكون حفرة صغيرة أو نتوء فجائي في منتصف المطاط من تلقاء نفسه!
* **معادلة لابل (Laplace's Equation):** هي المعادلة التي تصف هذا الاتزان التام وتمنع تكون أي حفر عشوائية.

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** لفهم كيف يستنتج الرياضيون حل معادلة لابل $(D_x^2 + D_y^2)z = 0$ باستخدام المعادلة المساعدة $m^2 + 1 = 0$ (صفحة 33 من المذكرة).
* **معضلة الانتقال إلى المريخ (The Scale-Up Dilemma):** الصيغة التحليلية $\phi_1(y + ix) + \phi_2(y - ix)$ ممتازة على ورقة بيضاء، لكن على المريخ توجد **10,000 بكسل عشوائي للحفر النيزكية**! لا توجد صيغة رياضية مغلقة واحدة يمكنها تحقيق 10,000 جدار حفرة، ولذلك سنحتاج في اللاب 6 لتحويل هذه المعادلة إلى شبكة رقمية 5-Point Stencil!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab4

%matplotlib inline
print("تم تهيئة بيئة اللاب 4 بنجاح! جاهز لحساب المؤثرات ومعادلة لابل.")


---
## 3. الوصفة الرياضية وحساب الحل المكمل والخاص (C.F. & P.I.)

1. **الدالة المكملة (Complementary Function - C.F.):**
   * نضع $D_x = m$ و $D_y = 1$ لتكوين المعادلة المساعدة $F(m, 1) = 0$.
   * إذا كانت الجذور حقيقية ومختلفة $m_1, m_2$: $z = \phi_1(y + m_1 x) + \phi_2(y + m_2 x)$.
   * إذا كانت الجذور متساوية $m_1 = m_2 = m$: $z = \phi_1(y + mx) + x \phi_2(y + mx)$.
   * إذا كانت مركبة $m = \pm i$ (معادلة لابل، ص 33): $z = \phi_1(y + ix) + \phi_2(y - ix)$.

2. **التكامل الخاص (Particular Integral - P.I.) لقوى الجاذبية والتذبذب:**
   * لقوة $\sin(ax + by)$: نستبدل $D_x^2 \to -a^2$ و $D_x D_y \to -ab$ و $D_y^2 \to -b^2$.


---
## 4. 📝 تحدي الورقة والقلم 1 (حل معادلة لابل من صفحة 33 بالمذكرة)

> 💡 *(إذا واجهتك صعوبة، راجع `INSTRUCTOR_ANSWER_KEY.md` القسم 4.1).*

**المسألة:** أوجد الجذور المميزة لمعادلة لابل:
$$(D_x^2 + D_y^2)z = 0$$
1. كون المعادلة المساعدة بوضع $D_x = m, D_y = 1$: $m^2 + 1 = 0$.
2. حل المعادلة لإيجاد الجذور $m_1, m_2$.

أدخل الجذور في بايثون (`1j` تعبر عن $i$ و `-1j` تعبر عن $-i$):


In [ ]:
# خلية تقييم: جذور معادلة لابل
### START CODE HERE ### (≈ 2 lines of code)
# أدخل الجذور التخيلية في بايثون باستخدام 1j و -1j
m1 = None
m2 = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_1(m1, m2)


---
## 5. 🚀 تحدي جسر مركبة المريخ (التكامل الخاص لقوة الجاذبية في الوادي)

في وادٍ مريخي شديد الانحدار، تضيف الجاذبية قوة متذبذبة $f(x, y) = \sin(2x - 3y)$:
$$(D_x^2 - D_x D_y - 6 D_y^2)z = \sin(2x - 3y)$$

**على ورقتك:**
1. باستخدام الحالة الثانية للتكامل الخاص (P.I. Case 2): استبدل $D_x^2 \to -(2)^2 = -4$ و $D_x D_y \to -(2)(-3) = 6$ و $D_y^2 \to -(-3)^2 = -9$.
2. احسب قيمة المؤثر: $F(-4, 6, -9) = -4 - 6 - 6(-9) = 44$.
3. اكتب التكامل الخاص $z_2 = \frac{1}{44}\sin(2x - 3y)$.

أدخل صيغة التكامل الخاص P.I. الناتجة أدناه:


In [ ]:
# خلية تقييم: التكامل الخاص P.I.
### START CODE HERE ### (≈ 1 line of code)
# اكتب صيغة التكامل الخاص (مثال: "(1/44)*sin(2x - 3y)")
student_pi = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_2(student_pi)


---
## 6. 💻 التطبيق البرمجي ببايثون: حل جذور مؤثرات الرتب العليا

سنكتب دالة تحسب جذور المعادلة المساعدة لأي مؤثر تفاضلي من الدرجة الثانية أو أعلى:


In [ ]:
# دالة تقييم برمجية: solve_operator_roots
def solve_operator_roots(coeffs: list[float]) -> list[complex]:
    # تحسب جذور المعادلة المساعدة F(m, 1) = 0
    ### START CODE HERE ### (≈ 1 line of code)
    # استخدم np.roots(coeffs)
    roots = None
    ### END CODE HERE ###
    return list(roots)

# تشغيل الاختبار الآلي:
test_lab4.test_characteristic_roots_solver(solve_operator_roots)


---
## 7. 🎨 الرؤية البصرية للفيزياء: سطح لابل التوافقي (انعدام الحفر تماماً!)
شغّل الخلية لرؤية الحل التحليلي التوافقي الدقيق $\phi(x, y) = \frac{\sin(\pi x)\sinh(\pi y)}{\sinh(\pi)}$:
لاحظ كيف ينحدر السطح بنعومة فائقة من أعلى نقطة للأسفل دون أي حفرة محلية تعيق الحركة!


In [ ]:
x = np.linspace(0, 1, 40)
y = np.linspace(0, 1, 40)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85)
ax.set_title("Harmonic Potential Surface $\nabla^2\phi = 0$: Zero Local Minima!", fontsize=13)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Potential $\phi$")
plt.colorbar(surf, shrink=0.5)
plt.show()

print("تهانينا! لقد أتممت اللاب 4 وأتقنت مؤثرات الرتب العليا وأصل معادلة لابل.")
